# 🧬 QGAN Drug Discovery — Classical MolGAN Baseline
**Phase 1 · Notebook 2 of 4**

Train the classical MolGAN baseline that the quantum generator will later replace.

### What this notebook does
1. Build the full MolGAN architecture (Generator + Discriminator + Reward Network)
2. Train with WGAN-GP + RL reward for 30 epochs
3. Plot loss curves and per-epoch validity / QED
4. Generate molecules and visualise results
5. Save baseline metrics to beat in Notebook 3

> **Run Notebook 1 first** — this notebook reuses the same dataset constants.


## 1 · Setup & Imports

In [ ]:
import warnings, os, time, pickle
warnings.filterwarnings('ignore')
os.makedirs('outputs/training', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

import numpy as np
import pandas as pd
from collections import Counter

# tqdm — works in both Jupyter and plain Python
try:
    from tqdm.notebook import tqdm
    from tqdm.notebook import trange
except ImportError:
    from tqdm import tqdm, trange

# scipy smoothing — optional
try:
    from scipy.ndimage import uniform_filter1d
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    def uniform_filter1d(arr, size=3):
        """Simple moving average fallback if scipy not installed."""
        result = []
        for i in range(len(arr)):
            lo = max(0, i - size//2)
            hi = min(len(arr), i + size//2 + 1)
            result.append(np.mean(arr[lo:hi]))
        return result

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR

from rdkit import Chem
from rdkit.Chem import QED, Descriptors, rdMolDescriptors

import matplotlib.pyplot as plt
import seaborn as sns

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = (torch.device('cuda')  if torch.cuda.is_available()  else
          torch.device('mps')   if torch.backends.mps.is_available() else
          torch.device('cpu'))
print(f"✓ Device        : {DEVICE}")
print(f"  PyTorch       : {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

# ── Constants ─────────────────────────────────────────────────────────────────
ATOM_TYPES     = ['C', 'N', 'O', 'F']
MAX_ATOMS      = 9
NUM_ATOM_TYPES = len(ATOM_TYPES) + 1   # +1 for PAD
NUM_BOND_TYPES = 5                      # none, single, double, triple, aromatic
Z_DIM          = 8
HIDDEN_G       = 128
HIDDEN_D       = 64
BATCH_SIZE     = 32
N_EPOCHS       = 30
N_CRITIC       = 5
LAMBDA_GP      = 10.0
LAMBDA_RL      = 1.0
LR_G           = 1e-4
LR_D           = 1e-4

BOND_ENCODER = {
    Chem.rdchem.BondType.SINGLE:   1,
    Chem.rdchem.BondType.DOUBLE:   2,
    Chem.rdchem.BondType.TRIPLE:   3,
    Chem.rdchem.BondType.AROMATIC: 4,
}

PALETTE = ['#6C63FF', '#FF6584', '#43B89C', '#FFA45B', '#A8DADC']
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False,
    'axes.spines.right': False, 'figure.facecolor': 'white',
})
print("✓ Constants set")
print(f"  Epochs={N_EPOCHS}, Batch={BATCH_SIZE}, z_dim={Z_DIM}, n_critic={N_CRITIC}")


## 2 · Dataset

In [ ]:
QM9_SMILES = [
    'C','CC','CCC','CCCC','CCCCC','C=C','C=CC','C#C','C#CC',
    'CC=C','CC=CC','CCC=C','C1CC1','C1CCC1','C1CCCC1','C1CCCCC1',
    'CO','CCO','CCCO','CCCCO','C=O','CC=O','CCC=O','C(=O)O',
    'CC(=O)O','CCC(=O)O','COC','CCOC','C1CO1','C1CCO1','C1CCCO1',
    'CN','CCN','CCCN','C=N','CC=N','C#N','CC#N','CCC#N',
    'C1CN1','C1CCN1','C1CCCN1','NC=O','NCC=O','NCN','NCCN',
    'CF','CCF','CCCF','FC=O','FCC=O','FCN','FCCN',
    'CC(N)=O','CC(O)=O','NCC(=O)O','OCC(=O)O','CC(=O)N',
    'CC(=O)NC','NCC#N','OCC#N','CC(O)C','CC(N)C',
    'C1=CC=CC=C1','C1=CN=CC=C1','C1=CC=NC=C1','C1=CC=CO1',
    'C1CCNCC1','C1CCNC1','C1CNCCN1','C1NCCN1',
    'OC1CCCCC1','NC1CCCCC1','CC1CCCCC1','CC1CCCC1',
    'CC(C)O','CC(C)N','CCC(O)=O','CCC(N)=O','CC(=O)OC',
    'CC(C)C=O','OC(CO)CO','NCC(O)CO','CC(C)(O)C','CC(C)(N)C',
    'C1CC(N)CC1','C1CC(O)CC1','OC1CCNCC1','NC1CCOC1',
    'NCC(=O)O','CC(N)C(=O)O','OCC(N)C(=O)O',
    'C=CC=C','C=CC#N','C=CN','C=CO','CC=CC=O','CC=CCN',
    'OC=O','OCC=O','NC(=O)C','NC(N)=O',
    'C1CC(=O)CC1','C1CC(N)CC1','C1CC(O)CCC1',
    'CC(F)=O','CC(F)F','C(F)(F)F',
]

# ── Molecule ↔ graph helpers ──────────────────────────────────────────────────
def mol_to_graph(mol):
    """RDKit mol → (adjacency [MAX_ATOMS,MAX_ATOMS], nodes [MAX_ATOMS])."""
    mol   = Chem.RemoveHs(mol)
    n     = mol.GetNumAtoms()
    if n > MAX_ATOMS: return None, None
    adj   = np.zeros((MAX_ATOMS, MAX_ATOMS), dtype=np.int64)
    nodes = np.zeros(MAX_ATOMS, dtype=np.int64)
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt   = BOND_ENCODER.get(bond.GetBondType(), 0)
        adj[i, j] = bt; adj[j, i] = bt
    for i, atom in enumerate(mol.GetAtoms()):
        sym      = atom.GetSymbol()
        nodes[i] = ATOM_TYPES.index(sym) if sym in ATOM_TYPES else len(ATOM_TYPES)
    return adj, nodes

def graph_to_mol(adj, nodes):
    """(adjacency, nodes) arrays → RDKit mol. Returns None if invalid."""
    mol      = Chem.RWMol()
    atom_map = {}
    for i, t in enumerate(nodes):
        if t < len(ATOM_TYPES):
            atom_map[i] = mol.AddAtom(Chem.Atom(ATOM_TYPES[t]))
    for i in range(MAX_ATOMS):
        for j in range(i+1, MAX_ATOMS):
            if adj[i, j] == 0 or i not in atom_map or j not in atom_map:
                continue
            bt = [Chem.rdchem.BondType.SINGLE,
                  Chem.rdchem.BondType.DOUBLE,
                  Chem.rdchem.BondType.TRIPLE,
                  Chem.rdchem.BondType.AROMATIC][adj[i,j]-1]
            mol.AddBond(atom_map[i], atom_map[j], bt)
    try:
        m = mol.GetMol(); Chem.SanitizeMol(m); return m
    except: return None

# ── Build dataset ─────────────────────────────────────────────────────────────
adjs_list, nodes_list = [], []
smiles_set = set()   # used later for novelty calculation

for smi in QM9_SMILES:
    mol = Chem.MolFromSmiles(smi)
    if not mol: continue
    mol  = Chem.RemoveHs(mol)
    csmi = Chem.MolToSmiles(mol)
    if csmi in smiles_set: continue
    adj, nds = mol_to_graph(mol)
    if adj is None: continue
    smiles_set.add(csmi)
    adjs_list.append(adj)
    nodes_list.append(nds)

adjs_np  = np.array(adjs_list,  dtype=np.int64)
nodes_np = np.array(nodes_list, dtype=np.int64)

class MolDataset(Dataset):
    def __init__(self, adjs, nodes):
        self.adjs  = torch.tensor(adjs,  dtype=torch.long)
        self.nodes = torch.tensor(nodes, dtype=torch.long)
    def __len__(self): return len(self.adjs)
    def __getitem__(self, i): return self.adjs[i], self.nodes[i]

dataset    = MolDataset(adjs_np, nodes_np)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, drop_last=True)

print(f"✓ Dataset ready")
print(f"  Molecules     : {len(dataset)}")
print(f"  Batches/epoch : {len(dataloader)}")
print(f"  smiles_set    : {len(smiles_set)} canonical SMILES (used for novelty)")

# ── Quick sanity: roundtrip one molecule ──────────────────────────────────────
test_adj, test_nodes = adjs_np[0], nodes_np[0]
test_mol = graph_to_mol(test_adj, test_nodes)
if test_mol:
    print(f"  Roundtrip OK  : {Chem.MolToSmiles(test_mol)}")
else:
    print("  ⚠ Roundtrip failed — check BOND_ENCODER")


## 3 · Model Architecture

In [ ]:
# ── Relational Graph Convolution ─────────────────────────────────────────────
class GraphConv(nn.Module):
    """Aggregates neighbour features per bond type (relational GCN)."""
    def __init__(self, in_dim, out_dim, num_relations=NUM_BOND_TYPES):
        super().__init__()
        self.linears  = nn.ModuleList([nn.Linear(in_dim, out_dim)
                                       for _ in range(num_relations)])
        self.self_lin = nn.Linear(in_dim, out_dim)
        self.bn       = nn.BatchNorm1d(out_dim)

    def forward(self, x, adj_oh):
        # x      : (B, N, in_dim)
        # adj_oh : (B, R, N, N)
        out = self.self_lin(x)
        for r, lin in enumerate(self.linears):
            out = out + lin(torch.bmm(adj_oh[:, r], x))
        B, N, D = out.shape
        out = self.bn(out.reshape(B*N, D)).reshape(B, N, D)
        return F.relu(out)

# ── Generator ─────────────────────────────────────────────────────────────────
class Generator(nn.Module):
    """Noise z → (adjacency logits, node logits)."""
    def __init__(self, z_dim=Z_DIM, hidden=HIDDEN_G):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),        nn.ReLU(),
            nn.Linear(hidden, hidden*2),     nn.ReLU(),
            nn.Linear(hidden*2, hidden*2),   nn.ReLU(),
        )
        self.adj_head  = nn.Linear(hidden*2, MAX_ATOMS*MAX_ATOMS*NUM_BOND_TYPES)
        self.node_head = nn.Linear(hidden*2, MAX_ATOMS*NUM_ATOM_TYPES)

    def forward(self, z):
        B = z.size(0); h = self.net(z)
        adj_logits  = self.adj_head(h).reshape(B, MAX_ATOMS, MAX_ATOMS, NUM_BOND_TYPES)
        node_logits = self.node_head(h).reshape(B, MAX_ATOMS, NUM_ATOM_TYPES)
        adj_logits  = (adj_logits + adj_logits.transpose(1,2)) / 2  # symmetrise
        return adj_logits, node_logits

    def sample(self, z):
        adj_l, node_l = self.forward(z)
        return adj_l.argmax(-1), node_l.argmax(-1)

# ── Discriminator ─────────────────────────────────────────────────────────────
class Discriminator(nn.Module):
    """Graph → real/fake scalar (WGAN, no sigmoid)."""
    def __init__(self, hidden=HIDDEN_D):
        super().__init__()
        self.embed = nn.Embedding(NUM_ATOM_TYPES+1, hidden)
        self.gcn1  = GraphConv(hidden, hidden)
        self.gcn2  = GraphConv(hidden, hidden)
        self.gcn3  = GraphConv(hidden, hidden)
        self.fc    = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                   nn.Linear(hidden, 1))

    def forward(self, adj, nodes):
        adj_oh = F.one_hot(adj, NUM_BOND_TYPES).float().permute(0,3,1,2)
        x = self.embed(nodes)
        x = self.gcn1(x, adj_oh)
        x = self.gcn2(x, adj_oh)
        x = self.gcn3(x, adj_oh)
        return self.fc(x.mean(dim=1)).squeeze(-1)

# ── Reward Network ────────────────────────────────────────────────────────────
class RewardNet(nn.Module):
    """Graph → drug-likeness score in [0,1]."""
    def __init__(self, hidden=HIDDEN_D):
        super().__init__()
        self.embed = nn.Embedding(NUM_ATOM_TYPES+1, hidden)
        self.gcn1  = GraphConv(hidden, hidden)
        self.gcn2  = GraphConv(hidden, hidden)
        self.fc    = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                   nn.Linear(hidden, 1), nn.Sigmoid())

    def forward(self, adj, nodes):
        adj_oh = F.one_hot(adj, NUM_BOND_TYPES).float().permute(0,3,1,2)
        x = self.embed(nodes)
        x = self.gcn1(x, adj_oh); x = self.gcn2(x, adj_oh)
        return self.fc(x.mean(dim=1)).squeeze(-1)

# ── Instantiate ───────────────────────────────────────────────────────────────
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
R = RewardNet().to(DEVICE)

def count_params(m): return sum(p.numel() for p in m.parameters())
print(f"✓ Models ready on {DEVICE}")
print(f"  Generator      : {count_params(G):,} parameters")
print(f"  Discriminator  : {count_params(D):,} parameters")
print(f"  Reward net     : {count_params(R):,} parameters")


## 4 · Training Utilities

In [ ]:
def sample_z(n): return torch.randn(n, Z_DIM, device=DEVICE)

def gradient_penalty(D, real_adj, real_nodes, fake_adj, fake_nodes):
    """WGAN-GP: penalise discriminator gradient norm at interpolated samples."""
    B     = real_adj.size(0)
    alpha = torch.rand(B, 1, 1, 1, device=DEVICE)
    real_oh = F.one_hot(real_adj, NUM_BOND_TYPES).float()
    fake_oh = F.one_hot(fake_adj, NUM_BOND_TYPES).float()
    interp  = (alpha * real_oh + (1-alpha) * fake_oh).requires_grad_(True)
    interp_idx = interp.argmax(-1)
    out = D(interp_idx, real_nodes)
    grads = torch.autograd.grad(
        outputs=out, inputs=interp,
        grad_outputs=torch.ones_like(out),
        create_graph=True, retain_graph=True, allow_unused=True
    )[0]
    if grads is None:
        return torch.tensor(0.0, device=DEVICE, requires_grad=True)
    gp = ((grads.reshape(B, -1).norm(2, dim=1) - 1)**2).mean()
    return gp

def evaluate_molecules(G, n_samples=256):
    """Generate n_samples molecules and compute validity / QED / uniqueness."""
    G.eval()
    with torch.no_grad():
        adj_b, node_b = G.sample(sample_z(n_samples))
    adj_b  = adj_b.cpu().numpy()
    node_b = node_b.cpu().numpy()

    valid_mols, valid_smiles = [], []
    for adj, nodes in zip(adj_b, node_b):
        mol = graph_to_mol(adj, nodes)
        if mol:
            smi = Chem.MolToSmiles(mol)
            valid_mols.append(mol); valid_smiles.append(smi)

    validity   = len(valid_mols) / n_samples
    uniqueness = len(set(valid_smiles)) / len(valid_smiles) if valid_smiles else 0.0

    qed_scores = []
    for mol in valid_mols:
        try: qed_scores.append(QED.qed(mol))
        except: pass

    return {
        'validity':   validity,
        'uniqueness': uniqueness,
        'qed_mean':   np.mean(qed_scores) if qed_scores else 0.0,
        'qed_std':    np.std(qed_scores)  if qed_scores else 0.0,
        'n_valid':    len(valid_mols),
        'n_unique':   len(set(valid_smiles)),
    }

print("✓ Utilities ready")
print("  gradient_penalty : WGAN-GP regularisation")
print("  evaluate_molecules: validity / uniqueness / QED")


### Optional: Quick 5-epoch test

In [ ]:
# ── OPTIONAL: 5-epoch quick test before full 30-epoch run ───────────────────
# Run this first to confirm the training loop works, then run Cell 10 (full run)

print("Running 5-epoch quick test...")
G_test = Generator().to(DEVICE)
D_test = Discriminator().to(DEVICE)
R_test = RewardNet().to(DEVICE)
opt_G_t = optim.Adam(G_test.parameters(), lr=LR_G, betas=(0.5, 0.9))
opt_D_t = optim.Adam(D_test.parameters(), lr=LR_D, betas=(0.5, 0.9))
opt_R_t = optim.Adam(R_test.parameters(), lr=1e-3)

for epoch in range(1, 6):
    G_test.train(); D_test.train(); R_test.train()
    for real_adj, real_nodes in dataloader:
        real_adj   = real_adj.to(DEVICE)
        real_nodes = real_nodes.to(DEVICE)
        B = real_adj.size(0)
        for _ in range(N_CRITIC):
            opt_D_t.zero_grad()
            with torch.no_grad():
                fa, fn = G_test.sample(sample_z(B))
            loss_d = (D_test(fa, fn).mean() - D_test(real_adj, real_nodes).mean()
                      + LAMBDA_GP * gradient_penalty(D_test, real_adj, real_nodes, fa, fn))
            loss_d.backward(); opt_D_t.step()
        opt_G_t.zero_grad()
        al, nl = G_test(sample_z(B)); fa = al.argmax(-1); fn = nl.argmax(-1)
        loss_g = -D_test(fa, fn).mean() - LAMBDA_RL * R_test(fa, fn).mean()
        loss_g.backward(); opt_G_t.step()
        opt_R_t.zero_grad()
        F.mse_loss(R_test(real_adj, real_nodes),
                   torch.full((B,), 0.5, device=DEVICE)).backward()
        opt_R_t.step()
    metrics = evaluate_molecules(G_test, 64)
    print(f"  Epoch {epoch} | D={loss_d.item():+.3f} | G={loss_g.item():+.3f} | "
          f"Valid={metrics['validity']:.1%} | QED={metrics['qed_mean']:.3f}")

del G_test, D_test, R_test
print("\n✓ Quick test passed — run Cell 10 for full 30-epoch training")


## 5 · Training Loop

In [ ]:
opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.9))
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.9))
opt_R = optim.Adam(R.parameters(), lr=1e-3)
sched_G = StepLR(opt_G, step_size=10, gamma=0.5)
sched_D = StepLR(opt_D, step_size=10, gamma=0.5)

# ── History tracking ──────────────────────────────────────────────────────────
history = {
    'epoch':      [],
    'loss_d':     [],  'loss_g':    [],
    'validity':   [],  'uniqueness':[],
    'qed_mean':   [],  'qed_std':   [],
    'n_valid':    [],
}

print(f"Starting training: {N_EPOCHS} epochs, batch={BATCH_SIZE}, device={DEVICE}")
print(f"Evaluating every epoch on 256 generated molecules")
print("─" * 65)

t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    G.train(); D.train(); R.train()
    epoch_loss_d, epoch_loss_g = [], []

    for real_adj, real_nodes in dataloader:
        real_adj   = real_adj.to(DEVICE)
        real_nodes = real_nodes.to(DEVICE)
        B          = real_adj.size(0)

        # ── Train Discriminator × N_CRITIC ───────────────────────────────────
        for _ in range(N_CRITIC):
            opt_D.zero_grad()
            with torch.no_grad():
                fake_adj, fake_nodes = G.sample(sample_z(B))
            d_real  = D(real_adj, real_nodes)
            d_fake  = D(fake_adj, fake_nodes)
            gp      = gradient_penalty(D, real_adj, real_nodes,
                                       fake_adj, fake_nodes)
            loss_d  = d_fake.mean() - d_real.mean() + LAMBDA_GP * gp
            loss_d.backward(); opt_D.step()

        epoch_loss_d.append(loss_d.item())

        # ── Train Generator ───────────────────────────────────────────────────
        opt_G.zero_grad()
        adj_logits, node_logits = G(sample_z(B))
        fake_adj   = adj_logits.argmax(-1)
        fake_nodes = node_logits.argmax(-1)
        d_fake     = D(fake_adj, fake_nodes)
        reward     = R(fake_adj, fake_nodes)

        # Entropy bonus keeps generator from collapsing
        ent_adj  = -(adj_logits.softmax(-1)  * adj_logits.log_softmax(-1)).sum(-1).mean()
        ent_node = -(node_logits.softmax(-1) * node_logits.log_softmax(-1)).sum(-1).mean()
        loss_g   = (-d_fake.mean()
                    - LAMBDA_RL * reward.mean()
                    - 0.01 * (ent_adj + ent_node))
        loss_g.backward(); opt_G.step()
        epoch_loss_g.append(loss_g.item())

        # ── Train Reward Net on real molecules (QED proxy = 0.5) ──────────────
        opt_R.zero_grad()
        r_pred   = R(real_adj, real_nodes)
        loss_r   = F.mse_loss(r_pred, torch.full_like(r_pred, 0.5))
        loss_r.backward(); opt_R.step()

    sched_G.step(); sched_D.step()

    # ── Evaluate ──────────────────────────────────────────────────────────────
    metrics = evaluate_molecules(G, n_samples=256)
    mean_d  = np.mean(epoch_loss_d)
    mean_g  = np.mean(epoch_loss_g)

    history['epoch'].append(epoch)
    history['loss_d'].append(mean_d)
    history['loss_g'].append(mean_g)
    history['validity'].append(metrics['validity'])
    history['uniqueness'].append(metrics['uniqueness'])
    history['qed_mean'].append(metrics['qed_mean'])
    history['qed_std'].append(metrics['qed_std'])
    history['n_valid'].append(metrics['n_valid'])

    elapsed = time.time() - t0
    print(f"Epoch {epoch:02d}/{N_EPOCHS} | "
          f"D={mean_d:+.3f} | G={mean_g:+.3f} | "
          f"Valid={metrics['validity']:.1%} | "
          f"Uniq={metrics['uniqueness']:.1%} | "
          f"QED={metrics['qed_mean']:.3f} | "
          f"⏱ {elapsed:.0f}s")

    # ── Save checkpoint every 10 epochs ───────────────────────────────────────
    if epoch % 10 == 0:
        torch.save({'epoch':epoch,'G':G.state_dict(),'D':D.state_dict(),
                    'R':R.state_dict(),'history':history},
                   f'checkpoints/molgan_epoch{epoch:03d}.pt')
        print(f"  💾 Checkpoint saved → checkpoints/molgan_epoch{epoch:03d}.pt")

print(f"\n✓ Training complete in {time.time()-t0:.1f}s")
torch.save({'epoch':N_EPOCHS,'G':G.state_dict(),'D':D.state_dict(),
            'R':R.state_dict(),'history':history},
           'checkpoints/molgan_final.pt')
print("  💾 Final model saved → checkpoints/molgan_final.pt")


## 6 · Loss Curves

### Load saved checkpoint (optional)

In [ ]:
# ── Load a saved checkpoint (skip if you just trained above) ─────────────────
# Uncomment these lines if you want to reload a saved model instead of retraining

# import pickle
# checkpoint = torch.load('checkpoints/molgan_final.pt', map_location=DEVICE)
# G.load_state_dict(checkpoint['G'])
# D.load_state_dict(checkpoint['D'])
# R.load_state_dict(checkpoint['R'])
# history = checkpoint['history']
# print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch']}")

# ── Or reload baseline metrics directly ───────────────────────────────────────
# with open('checkpoints/baseline_metrics.pkl', 'rb') as f:
#     baseline_metrics = pickle.load(f)
# print(baseline_metrics)

print("Checkpoint loader cell — uncomment lines above to reload a saved model.")
print("If you just ran the training cell, skip this cell.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MolGAN Training Curves', fontsize=15, fontweight='bold')
epochs = history['epoch']

# 6a — D and G losses
axes[0].plot(epochs, history['loss_d'], color=PALETTE[1], lw=2, label='Discriminator')
axes[0].plot(epochs, history['loss_g'], color=PALETTE[0], lw=2, label='Generator')
axes[0].axhline(0, color='gray', ls='--', lw=0.8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('WGAN Losses')
axes[0].legend()

# 6b — Validity and uniqueness
axes[1].plot(epochs, [v*100 for v in history['validity']],
             color=PALETTE[2], lw=2, marker='o', ms=4, label='Validity %')
axes[1].plot(epochs, [u*100 for u in history['uniqueness']],
             color=PALETTE[3], lw=2, marker='s', ms=4, label='Uniqueness %')
axes[1].axhline(98, color=PALETTE[2], ls='--', lw=1, alpha=0.5, label='MolGAN target (98%)')
axes[1].axhline(94, color=PALETTE[3], ls='--', lw=1, alpha=0.5, label='MolGAN target (94%)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('%')
axes[1].set_title('Validity & Uniqueness')
axes[1].legend(fontsize=8); axes[1].set_ylim(0, 105)

# 6c — QED mean ± std
qed_arr = np.array(history['qed_mean'])
qed_std = np.array(history['qed_std'])
axes[2].plot(epochs, qed_arr, color=PALETTE[4], lw=2, marker='o', ms=4, label='QED mean')
axes[2].fill_between(epochs, qed_arr-qed_std, qed_arr+qed_std,
                      alpha=0.2, color=PALETTE[4])
axes[2].axhline(0.47, color='red', ls='--', lw=1.2, label='MolGAN paper (0.47)')
axes[2].axhline(0.40, color='orange', ls='--', lw=1.2, label='Training mean (0.40)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('QED score')
axes[2].set_title('Drug-Likeness (QED) over Training')
axes[2].legend(fontsize=8); axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('outputs/training/01_loss_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved → outputs/training/01_loss_curves.png")


## 7 · Detailed Validity Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Validity Analysis Over Training', fontsize=14, fontweight='bold')

epochs = history['epoch']

# Valid count bar chart
axes[0].bar(epochs, history['n_valid'],
            color=PALETTE[0], alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Valid molecules (out of 256)')
axes[0].set_title('Valid molecules per evaluation')
for i, v in enumerate(history['n_valid']):
    if v > 0:
        axes[0].text(epochs[i], v+1, str(v),
                     ha='center', va='bottom', fontsize=7)

# Validity % with smoothed trend
val_arr    = history['validity']
val_smooth = uniform_filter1d(val_arr, size=3)   # works with scipy or fallback

axes[1].plot(epochs, [v*100 for v in val_arr],
             color=PALETTE[0], alpha=0.35, lw=1.5, label='Raw')
axes[1].plot(epochs, [v*100 for v in val_smooth],
             color=PALETTE[0], lw=2.5, label='Smoothed (window=3)')
axes[1].axhline(40, color='green',  ls='--', lw=1.2, label='Good target (40%)')
axes[1].axhline(20, color='orange', ls='--', lw=1.2, label='Acceptable (20%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validity %')
axes[1].set_title('Validity % over Training')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig('outputs/training/02_validity_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

# ── Training history table ────────────────────────────────────────────────────
df_hist = pd.DataFrame(history)
print("\n── Training History ──────────────────────────────────")
print(df_hist[['epoch','loss_d','loss_g','validity',
               'uniqueness','qed_mean','n_valid']]
      .to_string(index=False, float_format=lambda x: f'{x:.3f}'))


## 8 · Final Model Evaluation

In [ ]:
# ── Generate 512 molecules for final evaluation ──────────────────────────────
G.eval()
with torch.no_grad():
    final_adj, final_nodes = G.sample(sample_z(512))
final_adj   = final_adj.cpu().numpy()
final_nodes = final_nodes.cpu().numpy()

valid_mols, valid_smiles = [], []
for adj, nodes in zip(final_adj, final_nodes):
    mol = graph_to_mol(adj, nodes)
    if mol:
        try:
            Chem.SanitizeMol(mol)
            smi = Chem.MolToSmiles(mol)
            valid_mols.append(mol); valid_smiles.append(smi)
        except: pass

validity   = len(valid_mols) / 512
uniqueness = len(set(valid_smiles)) / len(valid_smiles) if valid_smiles else 0
novelty    = len(set(valid_smiles) - smiles_set) / len(set(valid_smiles)) if valid_smiles else 0

# Property scores
qed_scores, logp_scores, mw_scores = [], [], []
for mol in valid_mols:
    try:
        qed_scores.append(QED.qed(mol))
        logp_scores.append(Descriptors.MolLogP(mol))
        mw_scores.append(Descriptors.MolWt(mol))
    except: pass

print("═" * 55)
print("  BASELINE MODEL — FINAL EVALUATION (512 molecules)")
print("═" * 55)
print(f"  Validity   : {validity:.1%}   ({len(valid_mols)}/512)")
print(f"  Uniqueness : {uniqueness:.1%}   ({len(set(valid_smiles))} unique)")
print(f"  Novelty    : {novelty:.1%}   (not in training set)")
if qed_scores:
    print(f"  QED mean   : {np.mean(qed_scores):.3f} ± {np.std(qed_scores):.3f}")
    print(f"  logP mean  : {np.mean(logp_scores):.2f}")
    print(f"  MW mean    : {np.mean(mw_scores):.1f} Da")
print("─" * 55)
print("  MolGAN paper targets:")
print("    Validity   : ~98%")
print("    Uniqueness : ~94%")
print("    QED        : ~0.47")
print("═" * 55)

# Save baseline metrics for Notebook 3 comparison
baseline_metrics = {
    'validity': validity, 'uniqueness': uniqueness, 'novelty': novelty,
    'qed_mean': np.mean(qed_scores) if qed_scores else 0,
    'qed_std':  np.std(qed_scores)  if qed_scores else 0,
    'logp_mean':np.mean(logp_scores) if logp_scores else 0,
    'mw_mean':  np.mean(mw_scores)   if mw_scores  else 0,
    'n_valid':  len(valid_mols),
}
import pickle
with open('checkpoints/baseline_metrics.pkl', 'wb') as f:
    pickle.dump(baseline_metrics, f)
print("\n  Baseline metrics saved → checkpoints/baseline_metrics.pkl")
print("  (Notebook 3 will load this for QGAN vs MolGAN comparison)")


## 9 · Property Distribution of Generated Molecules

In [ ]:
if len(valid_mols) == 0:
    print("No valid molecules generated — try training for more epochs.")
else:
    # Build dataframe of generated molecules
    gen_records = []
    for mol, smi in zip(valid_mols, valid_smiles):
        try:
            gen_records.append({
                'smiles': smi,
                'qed':    QED.qed(mol),
                'logp':   Descriptors.MolLogP(mol),
                'mw':     Descriptors.MolWt(mol),
                'n_atoms':mol.GetNumAtoms(),
                'n_rings':rdMolDescriptors.CalcNumRings(mol),
                'novel':  smi not in smiles_set,
            })
        except: pass
    df_gen = pd.DataFrame(gen_records).drop_duplicates('smiles').reset_index(drop=True)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle('Generated Molecule Properties — Baseline MolGAN',
                 fontsize=15, fontweight='bold')

    # QED
    axes[0,0].hist(df_gen['qed'], bins=15, color=PALETTE[0], alpha=0.8, edgecolor='white',
                   label='Generated', density=True)
    axes[0,0].axvline(np.mean(qed_scores), color='red', ls='--', lw=1.8,
                       label=f"Gen mean: {np.mean(qed_scores):.3f}")
    axes[0,0].axvline(0.405, color='green', ls=':', lw=1.8, label='Training mean: 0.405')
    axes[0,0].set_xlabel('QED score'); axes[0,0].set_title('QED distribution')
    axes[0,0].legend(fontsize=8); axes[0,0].set_xlim(0, 1)

    # logP
    axes[0,1].hist(df_gen['logp'], bins=15, color=PALETTE[1], alpha=0.8, edgecolor='white')
    axes[0,1].axvline(5, color='red', ls='--', lw=1.5, label='Lipinski limit')
    axes[0,1].set_xlabel('logP'); axes[0,1].set_title('logP distribution')
    axes[0,1].legend()

    # MW
    axes[0,2].hist(df_gen['mw'], bins=15, color=PALETTE[2], alpha=0.8, edgecolor='white')
    axes[0,2].set_xlabel('MW (Da)'); axes[0,2].set_title('Molecular weight')

    # Atom count
    nc = df_gen['n_atoms'].value_counts().sort_index()
    axes[1,0].bar(nc.index, nc.values, color=PALETTE[3], alpha=0.8, edgecolor='white')
    axes[1,0].set_xlabel('# heavy atoms'); axes[1,0].set_title('Atom count')

    # Novel vs known
    novel_ct = df_gen['novel'].value_counts()
    axes[1,1].pie([novel_ct.get(True,0), novel_ct.get(False,0)],
                   labels=[f"Novel\n({novel_ct.get(True,0)})",
                           f"Known\n({novel_ct.get(False,0)})"],
                   colors=[PALETTE[2], PALETTE[4]], autopct='%1.0f%%',
                   startangle=90, wedgeprops=dict(edgecolor='white',linewidth=2))
    axes[1,1].set_title('Novel vs Training Set')

    # QED vs logP scatter
    sc = axes[1,2].scatter(df_gen['logp'], df_gen['qed'],
                            c=df_gen['n_rings'], cmap='plasma', alpha=0.7, s=50)
    plt.colorbar(sc, ax=axes[1,2], label='# rings')
    axes[1,2].axvline(5,   color='red',    ls='--', lw=1, alpha=0.5)
    axes[1,2].axhline(0.4, color='orange', ls='--', lw=1, alpha=0.5)
    axes[1,2].set_xlabel('logP'); axes[1,2].set_ylabel('QED')
    axes[1,2].set_title('QED vs logP  (colour = rings)')

    plt.tight_layout()
    plt.savefig('outputs/training/03_generated_properties.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Generated {len(df_gen)} unique valid molecules")
    print("Saved → outputs/training/03_generated_properties.png")


## 10 · Summary & Next Steps

### What we built
- **Generator** — 3-layer MLP, noise z → adjacency + node logits  
- **Discriminator** — 3-layer relational GCN, WGAN (no sigmoid)  
- **Reward network** — 2-layer relational GCN, drug-likeness proxy  
- **Training** — WGAN-GP + RL reward + entropy regularisation  

### Baseline metrics saved
These are loaded in Notebook 3 to compare QGAN vs classical MolGAN:

```python
import pickle
with open('checkpoints/baseline_metrics.pkl', 'rb') as f:
    baseline = pickle.load(f)
```

### Interpreting results
| Validity | Meaning |
|---|---|
| < 10% | Model still learning graph structure |
| 10–40% | Reasonable — early-stage training |
| 40–70% | Good — model learning chemistry |
| > 70% | Excellent — approaching MolGAN paper results |

Low validity on a small dataset (95 molecules) is **expected** — the real QM9 has 134k molecules. The architecture is correct; scaling the dataset is the key lever.

### Next: Notebook 3 — Quantum Generator
Replace the classical MLP generator with a **PennyLane parameterized quantum circuit**:
- 9 qubits → one per atom position  
- Hadamard → CNOT entanglement → RY/RZ rotation layers  
- Same discriminator and reward network (unchanged)  
- Compare QGAN vs MolGAN on validity / QED / novelty  
